<a href="https://colab.research.google.com/github/DQ0115/1142programing_language/blob/main/HW4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：PTT → Google Sheet → RAG（整理版）

這份 notebook 保留完整流程：

1. 爬取 PTT NSwitch 文章
2. 寫入指定 Google Sheet
3. 從 Google Sheet 讀回資料
4. 建立 FAISS RAG 索引
5. 用 Gemini 根據 PTT 資料回答問題

主要修正：原本設定了 `SHEET_URL`，但實際用 `gc.open(WORKSHEET_NAME)` 開啟試算表，容易打開錯的 Spreadsheet。新版固定使用 `gc.open_by_url(SHEET_URL)`。


In [1]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai


In [2]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。  
`PTT_WORKSHEET_NAME` 是存放 PTT 原始文章的分頁。


In [3]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1nCVwpdHJrqfwtOmBeht4Dy8zXR8vwIvxNS1FcRcrR0U/edit?usp=sharing"
PTT_WORKSHEET_NAME = "ptt_games"
GEMINI_WORKSHEET_NAME = "gemini_answers"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]

GEMINI_HEADER = [
    "timestamp", "question", "answer"
]

PTT_NSwitch_INDEX = "https://www.ptt.cc/bbs/NSwitch/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (compatible; Colab PTT crawler)"

## 2. 連線 Google Sheet

這裡是最重要的修正：使用 `open_by_url(SHEET_URL)`，不要用 worksheet 名稱打開 spreadsheet。


In [4]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


✅ 已開啟試算表：HW4
🔗 https://docs.google.com/spreadsheets/d/1nCVwpdHJrqfwtOmBeht4Dy8zXR8vwIvxNS1FcRcrR0U/edit?usp=sharing


In [5]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
        # 若你要保留舊資料，請先備份 Google Sheet。
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)


ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)
ws_gemini_answers = ensure_worksheet(sh, GEMINI_WORKSHEET_NAME, GEMINI_HEADER)
print(f"✅ 已準備 worksheet：{ws_ptt.title}")
print(f"✅ 已準備 worksheet：{ws_gemini_answers.title}")

✅ 已準備 worksheet：ptt_games
✅ 已準備 worksheet：gemini_answers


## 3. PTT NSwitch 爬蟲

這段只負責爬 PTT，不碰 RAG。資料會先存在 `new_posts_df`。


In [6]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_soup(url):
    resp = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": USER_AGENT},
        cookies=PTT_COOKIES,
    )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆":
        return 100
    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10
    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a:
            continue

        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))
        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")

        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main:
        return "", ""

    # 取出文章建立時間
    created_at = ""
    metalines = main.select("div.article-metaline")
    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    # 移除 meta 與推文
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at


def make_post_id(url):
    # 用文章網址檔名當 post_id，穩定且方便去重
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_NSwitch(pages=1, delay=0.5):
    """爬取 PTT NSwitch 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_NSwitch_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }
                all_rows.append(row)
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")

        prev_url = get_prev_index_url(index_soup)
        if not prev_url:
            break
        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次爬到 {len(df)} 篇文章")
    return df

## 4. 執行爬蟲並寫入 Google Sheet

這一格會：

1. 從 Google Sheet 讀取既有資料
2. 爬取新的 PTT 資料
3. 合併並用 `post_id` 去重
4. 寫回 Google Sheet
5. 再讀一次確認真的寫入成功


In [8]:
# 你可以調整 pages，例如 pages=1 先測試，確認成功後再改成 3 或 5
new_posts_df = crawl_ptt_NSwitch(pages=1, delay=1.0)

old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)

written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")

📄 正在讀取列表頁 1/1: https://www.ptt.cc/bbs/NSwitch/index.html
✅ 本次爬到 4 篇文章
📌 Google Sheet 原本有 0 筆
✅ 已寫入 Google Sheet：4 筆
🔍 從 Google Sheet 重新讀回：4 筆
✅ 寫入驗證成功


## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`PTT → Google Sheet → RAG`


In [9]:
# 從 Google Sheet 重新讀取，作為 RAG 的唯一資料來源
rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)

# 清掉沒有內容的文章
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

rag_source_df.head()


📚 可用於 RAG 的文章數：4


,post_id,title,url,date,author,nrec,created_at,fetched_at,content
0,M.1633087289.A.64A,[公告] 常駐置底 情報/聊天/約戰/報價 區,https://www.ptt.cc/bbs/NSwitch/M.1633087289.A....,10/01,swallow0327,68,Fri Oct 1 19:21:26 2021,2026-06-01 10:53:29,本區功能範圍\n各式疑難雜症詢問\n遊戲推薦/反推\n特價/優惠分享\n臨時約戰\n閒聊\n...
1,M.1743609861.A.2C8,[公告] NSwitch 板規 Ver 6.0.0,https://www.ptt.cc/bbs/NSwitch/M.1743609861.A....,04/03,swallow0327,0,Thu Apr 3 00:04:18 2025,2026-06-01 10:53:27,本板宗旨\n本板為\n任天堂主機板\n發表文章需以\nNintendo Switch系列\n...
2,M.1524967710.A.D8A,[公告] FC板友交流徵家人長期約戰置底區,https://www.ptt.cc/bbs/NSwitch/M.1524967710.A....,4/29,nicetree,100,Sun Apr 29 10:08:22 2018,2026-06-01 10:53:26,各位板友好:\n\n鑒於任天堂官方已有越來越多網路遊戲功能開啟且開放線上付費及家人團報\n\...
3,M.1780290193.A.25E,[情報] 任天堂證實新款混合主機將登場,https://www.ptt.cc/bbs/NSwitch/M.1780290193.A....,06/01,pelchen,-1,Mon Jun 1 13:03:06 2026,2026-06-01 10:53:23,https://sunmedia.tw/news/Industry-information/...


In [10]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")


def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))

        text = (f"標題：{title}\n"
                f"作者：{author}\n"
                f"日期：{date}\n"
                f"推文數：{nrec}\n"
                f"內容：{content}")
        docs.append({
            "post_id": str(row.get("post_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding 模型載入完成


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RAG 索引建立完成：4 篇文章，向量維度 384


## 6. Gemini 設定與 RAG 問答

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。


In [11]:
api_key = userdata.get("gemini")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 若你的帳號不支援這個模型，可改成你可用的 Gemini model name
GEMINI_MODEL_NAME = "gemini-3-flash-preview"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


✅ Gemini 已設定：gemini-3-flash-preview


In [12]:
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據 PTT NSwitch版資料回答問題的幫手。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    # 移除 timeout 參數，因為它導致了 GenerativeModel.generate_content() 的錯誤
    response = llm.generate_content(prompt)
    return response.text

## 7. 快速測試


In [15]:
question = input("請輸入問題：")
answer = query_rag(question, k=3)
print(answer)


請輸入問題：今天天氣如何?
目前資料不足，無法判斷。

參考來源：
1. [公告] 常駐置底 情報/聊天/約戰/報價 區
https://www.ptt.cc/bbs/NSwitch/M.1633087289.A.64A.html
2. [公告] NSwitch 板規 Ver 6.0.0
https://www.ptt.cc/bbs/NSwitch/M.1743609861.A.2C8.html
3. [公告] FC板友交流徵家人長期約戰置底區
https://www.ptt.cc/bbs/NSwitch/M.1524967710.A.D8A.html


## 8. 新增記事儲存功能

你可以點擊下方的按鈕，將 AI 的回答儲存到 Google Sheet 的 `gemini_answers` 分頁中。

In [14]:
def save_gemini_answer(question, answer):
    df_gemini = read_sheet_df(ws_gemini_answers, GEMINI_HEADER)

    new_row = pd.DataFrame({
        "timestamp": [now_iso()],
        "question": [question],
        "answer": [answer]
    })

    df_gemini = pd.concat([df_gemini, new_row], ignore_index=True)
    written_count = write_sheet_df(ws_gemini_answers, df_gemini, GEMINI_HEADER)
    print(f"✅ 已儲存問題與回答至 Google Sheet：{written_count} 筆")


# 為了讓按鈕可以互動，需要導入 ipywidgets
from ipywidgets import Button, Output
from IPython.display import display

save_button = Button(description="儲存至記事")
output = Output()

def on_button_clicked(b):
    with output:
        if 'question' in globals() and 'answer' in globals():
            save_gemini_answer(question, answer)
        else:
            print("⚠️ 無法儲存，請先執行上面的問題與回答單元。")

save_button.on_click(on_button_clicked)
display(save_button, output)

Button(description='儲存至記事', style=ButtonStyle())

Output()

## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet，而不是記憶體中的暫存變數。
